In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, RandomizedSearchCV

from sklearn.neural_network import MLPClassifier

from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler

from scipy.stats import uniform, randint # для рандомности нейронов

In [2]:
# DataFrame (используем вашу)
np.random.seed(42)
num_clients = 300
income = np.random.randint(30000, 100000, num_clients)
credit_history = np.random.rand(num_clients)
probability_of_loan = 0.5 * income + 10000 * credit_history + np.random.normal(0, 5000, num_clients)
loan_decision = (probability_of_loan > np.mean(probability_of_loan)).astype(int)
data = pd.DataFrame({'income': income, 'credit_history': credit_history, 'loan_decision': loan_decision})
data

,income,credit_history,loan_decision
0,45795,0.590833,0
1,30860,0.030500,0
2,84886,0.037348,1
3,36265,0.822601,0
4,67194,0.360191,1
...,...,...,...
295,40966,0.633401,0
296,82921,0.240146,1
297,79726,0.075863,0
298,80300,0.128880,1


In [3]:
#  Подготовка данных
X = data[['income', 'credit_history']]
y = data['loan_decision']

# 3. Разделение на обучающую и тестовую выборки
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [4]:
#  Масштабирование признаков (ОЧЕНЬ ВАЖНО для MLP!)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# MLP

In [5]:
#  Создание и обучение MLP
mlp = MLPClassifier(
    hidden_layer_sizes=(10, ),  # Один скрытый слой с 10 нейронами
    activation='relu',
    solver='adam',
    alpha=0.0001,
    batch_size='auto',
    learning_rate_init=0.001,
    max_iter=200,
    random_state=42
)

mlp.fit(X_train_scaled, y_train)


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


,hidden_layer_sizes,"(10,)"
,activation,'relu'
,solver,'adam'
,alpha,0.0001
,batch_size,'auto'
,learning_rate,'constant'
,learning_rate_init,0.001
,power_t,0.5
,max_iter,200
,shuffle,True
,random_state,42


In [6]:
# Оценка модели
y_pred_mlp_t = mlp.predict(X_train_scaled)
y_pred_mlp = mlp.predict(X_test_scaled)

accuracy_mlp_t = accuracy_score(y_train, y_pred_mlp_t)
accuracy_mlp = accuracy_score(y_test, y_pred_mlp)

print(f"MLP Accuracy train: {accuracy_mlp_t:.2f}")
print(f"MLP Accuracy test: {accuracy_mlp:.2f}")

print("\nClassification Report (train):\n", classification_report(y_train, y_pred_mlp_t))
print("\nClassification Report (test):\n", classification_report(y_test, y_pred_mlp))

MLP Accuracy train: 0.90
MLP Accuracy test: 0.93

Classification Report (train):
               precision    recall  f1-score   support

           0       0.91      0.89      0.90       118
           1       0.90      0.92      0.91       122

    accuracy                           0.90       240
   macro avg       0.90      0.90      0.90       240
weighted avg       0.90      0.90      0.90       240


Classification Report (test):
               precision    recall  f1-score   support

           0       1.00      0.86      0.93        29
           1       0.89      1.00      0.94        31

    accuracy                           0.93        60
   macro avg       0.94      0.93      0.93        60
weighted avg       0.94      0.93      0.93        60



In [10]:
# 7. Предсказание для нового клиента
new_income = 60000
new_credit_history = 0.01

new_client = pd.DataFrame({'income': [new_income], 'credit_history': [new_credit_history]})
new_client_scaled = scaler.transform(new_client)

loan_decision_new_client = mlp.predict(new_client_scaled)[0]
probability_of_loan_new_client = mlp.predict_proba(new_client_scaled)[0, 1] #вероятность

print(f"\nПредсказание для нового клиента:")
print(f"Доход: {new_income}, Кредитная история: {new_credit_history}")
print(f"Вероятность выдачи кредита: {probability_of_loan_new_client:.2f}")
print(f"Решение о выдаче кредита (1 - выдать, 0 - отказать): {loan_decision_new_client}")


Предсказание для нового клиента:
Доход: 60000, Кредитная история: 0.01
Вероятность выдачи кредита: 0.32
Решение о выдаче кредита (1 - выдать, 0 - отказать): 0


 # MPL с  RandomizedSearchCV

In [11]:
#  RandomizedSearchCV для MLP
param_distributions = {
    'hidden_layer_sizes': [(randint(10, 100).rvs(1)[0],),  # Один скрытый слой, случайное количество нейронов
                            (randint(10, 100).rvs(1)[0], randint(10, 100).rvs(1)[0],)],  # Два скрытых слоя, случайное количество нейронов
    'activation': ['relu', 'tanh', 'logistic'],
    'solver': ['adam', 'lbfgs'],
    'alpha': uniform(0.0001, 0.01),
    'learning_rate_init': uniform(0.0001, 0.01),
    'max_iter': [200, 300, 400]
}

mlp = MLPClassifier(random_state=42)  # Создаем объект MLP
random_search = RandomizedSearchCV(
    mlp,
    param_distributions=param_distributions,
    n_iter=10,  # Количество случайных комбинаций параметров для перебора
    scoring='accuracy',
    cv=3,
    random_state=42,
    verbose=0,
    n_jobs=-1 #Использовать все ядра процессора
)

random_search.fit(X_train_scaled, y_train)

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:602: ConvergenceWarning: lbfgs failed to converge after 200 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=200).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:602: ConvergenceWarning: lbfgs failed to converge after 200 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=200).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_opt

,estimator,MLPClassifier(random_state=42)
,param_distributions,"{'activation': ['relu', 'tanh', ...], 'alpha': <scipy.stats....t 0x1150fb230>, 'hidden_layer_sizes': [(np.int64(64),), (np.int64(33), ...)], 'learning_rate_init': <scipy.stats....t 0x115119d10>, ...}"
,n_iter,10
,scoring,'accuracy'
,n_jobs,-1
,refit,True
,cv,3
,verbose,0
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [14]:
# Лучшая модель
best_mlp = random_search.best_estimator_
print("Лучшие параметры для MLP:", random_search.best_params_)

Лучшие параметры для MLP: {'activation': 'logistic', 'alpha': np.float64(0.008065429868602328), 'hidden_layer_sizes': (np.int64(64),), 'learning_rate_init': np.float64(0.007419939418114052), 'max_iter': 200, 'solver': 'adam'}


In [15]:
# Оценка модели
y_pred_mlp_t = best_mlp.predict(X_train_scaled)
y_pred_mlp = best_mlp.predict(X_test_scaled)

accuracy_mlp_t = accuracy_score(y_train, y_pred_mlp_t)
accuracy_mlp = accuracy_score(y_test, y_pred_mlp)

print(f"MLP Accuracy train: {accuracy_mlp_t:.2f}")
print(f"MLP Accuracy test: {accuracy_mlp:.2f}")

print("\nClassification Report (train):\n", classification_report(y_train, y_pred_mlp_t))
print("\nClassification Report (test):\n", classification_report(y_test, y_pred_mlp))


MLP Accuracy train: 0.90
MLP Accuracy test: 0.95

Classification Report (train):
               precision    recall  f1-score   support

           0       0.89      0.91      0.90       118
           1       0.91      0.89      0.90       122

    accuracy                           0.90       240
   macro avg       0.90      0.90      0.90       240
weighted avg       0.90      0.90      0.90       240


Classification Report (test):
               precision    recall  f1-score   support

           0       0.96      0.93      0.95        29
           1       0.94      0.97      0.95        31

    accuracy                           0.95        60
   macro avg       0.95      0.95      0.95        60
weighted avg       0.95      0.95      0.95        60



In [16]:
#  Предсказание для нового клиента
new_income = 60000
new_credit_history = 0.7

new_client = pd.DataFrame({'income': [new_income], 'credit_history': [new_credit_history]})
new_client_scaled = scaler.transform(new_client)

loan_decision_new_client = best_mlp.predict(new_client_scaled)[0]
probability_of_loan_new_client = best_mlp.predict_proba(new_client_scaled)[0, 1]

print(f"\nПредсказание для нового клиента:")
print(f"Доход: {new_income}, Кредитная история: {new_credit_history}")
print(f"Вероятность выдачи кредита: {probability_of_loan_new_client:.2f}")
print(f"Решение о выдаче кредита (1 - выдать, 0 - отказать): {loan_decision_new_client}")


Предсказание для нового клиента:
Доход: 60000, Кредитная история: 0.7
Вероятность выдачи кредита: 0.42
Решение о выдаче кредита (1 - выдать, 0 - отказать): 0
